In [2]:
import vertexai

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import google_search
from vertexai.preview.reasoning_engines import AdkApp
from vertexai import agent_engines

PROJECT_ID = "qwiklabs-gcp-02-64fe8ee0c5bc"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://qwiklabs-gcp-02-64fe8ee0c5bc-agent-staging"
MODEL = "gemini-2.5-flash"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print("Vertex AI initialized.")

Vertex AI initialized.


In [3]:
greeter_agent = LlmAgent(
    name="greeter_agent",
    model=MODEL,
    description="Greets the user and restates the request for the workflow.",
    instruction="""
    You are the greeter for a travel planning workflow.

    Briefly acknowledge the user's request and restate what information
    the workflow should research.

    Keep the response concise.
    """,
    output_key="greeter_output",
)

search_agent = LlmAgent(
    name="search_agent",
    model=MODEL,
    description="Uses Google Search to research the user's travel question.",
    instruction="""
    You are the research agent in a travel planning workflow.

    Use Google Search to gather useful information that answers
    the user's request.

    Base your response on the search results.

    Provide a clear initial answer that can later be reviewed
    and improved by other agents in the workflow.
    """,
    tools=[google_search],
    output_key="search_output",
)

critique_agent = LlmAgent(
    name="critique_agent",
    model=MODEL,
    description="Reviews the initial travel answer and recommends improvements.",
    instruction="""
    You are a critical reviewer in a travel planning workflow.

    Review the initial answer below:

    {search_output}

    Identify specific ways the answer could be improved.

    Consider:
    - relevance to the user's request
    - organization and clarity
    - whether the answer is too long or repetitive
    - whether the recommendations are practical for a weekend trip
    - whether important caveats or useful planning details are missing

    Do not rewrite the answer yet.
    Provide concise, actionable critique for the Refine Agent.
    """,
    output_key="critique_output",
)

refine_agent = LlmAgent(
    name="refine_agent",
    model=MODEL,
    description="Refines the initial travel answer using the critique.",
    instruction="""
    You are the final editor in a travel planning workflow.

    Initial answer:

    {search_output}

    Critique:

    {critique_output}

    Rewrite the initial answer using the critique.

    Requirements:
    - Keep the most useful recommendations.
    - Make the answer practical for a weekend trip.
    - Improve organization and clarity.
    - Remove unnecessary or repetitive information.
    - Include useful planning details when appropriate.
    - Do not mention the critique or the workflow in the final answer.
    - Return only the polished final response.
    """,
    output_key="refined_output",
)

In [4]:
answer_workflow = SequentialAgent(
    name="answer_workflow",
    description=(
        "Greets the user, researches the request, critiques the initial "
        "answer, and produces a refined final response."
    ),
    sub_agents=[
        greeter_agent,
        search_agent,
        critique_agent,
        refine_agent,
    ],
)

print("Challenge 4 workflow created.")

Challenge 4 workflow created.


/tmp/ipykernel_97919/939550003.py:1: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_workflow = SequentialAgent(


In [6]:
from vertexai.preview.reasoning_engines import AdkApp

app = AdkApp(
    agent=answer_workflow,
)

print("ADK app created.")

ADK app created.


## Local Workflow Test

Test the complete Greeter → Search → Critique → Refine workflow locally
through `AdkApp` before deploying it to Agent Platform.

In [7]:
LOCAL_USER_ID = "challenge5-clean-local-test"

local_session = app.create_session(
    user_id=LOCAL_USER_ID
)

print(f"Session ID: {local_session['id']}")
print("Running local workflow test...\n")

for event in app.stream_query(
    user_id=LOCAL_USER_ID,
    session_id=local_session["id"],
    message="I am planning a weekend trip to Denver. What are some good things to do?",
):
    print(event)

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


Session ID: 582e3878-a2bb-44a4-aebd-51cc07e93d9a
Running local workflow test...

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': "Hello! I understand you're planning a weekend trip to Denver and are looking for things to do there. This workflow will help you find some good activities and attractions for your trip."}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 37, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 37}], 'prompt_token_count': 96, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 96}], 'thoughts_token_count': 43, 'total_token_count': 176, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -0.7987953392235009, 'invocation_id': 'e-16a103dc-236a-48f4-9c04-aebcd5c3b143', 'author': 'greeter_agent', 'actions': {'state_delta': {'greeter_output': "Hello! I understand you're planning a weekend trip to Denver and are looking for things to do there. This workflow will help you find some good act

In [8]:
from vertexai import agent_engines

print("Deploying clean Challenge 4 workflow...")

remote_agent = agent_engines.create(
    app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]==1.163.0",
        "google-adk==2.4.0",
        "cloudpickle==3.1.2",
        "pydantic==2.13.4",
    ],
)

print("\nClean workflow deployment completed.")
print(remote_agent)

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.13.4', 'google-cloud-aiplatform': '1.163.0', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==1.163.0', 'google-adk==2.4.0', 'cloudpickle==3.1.2', 'pydantic==2.13.4']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-02-64fe8ee0c5bc-agent-staging


Deploying clean Challenge 4 workflow...


INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-02-64fe8ee0c5bc-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-02-64fe8ee0c5bc-agent-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-02-64fe8ee0c5bc-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/438227084378/locations/us-central1/reasoningEngines/6635215673314246656/operations/116601580333039616
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-02-64fe8ee0c5bc
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/438227084378/locations/us-central1/reasoningEngines/6635215673314246656
INFO:vertexai.agent_engines:To use this AgentEngine in another session:
INF


Clean workflow deployment completed.
resource name: projects/438227084378/locations/us-central1/reasoningEngines/6635215673314246656


In [9]:
REMOTE_USER_ID = "challenge5-clean-remote-test"

print("Sending request to clean deployed workflow...")

event_count = 0

async for event in remote_agent.async_stream_query(
    user_id=REMOTE_USER_ID,
    message="I am planning a weekend trip to Denver. What are some good things to do?",
):
    event_count += 1
    print(f"\nEVENT {event_count}:")
    print(event)

print(f"\nRemote test completed. Events received: {event_count}")

Sending request to clean deployed workflow...

EVENT 1:
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': "Understood! I'll research good things to do for a weekend trip to Denver."}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 18, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 18}], 'prompt_token_count': 96, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 96}], 'thoughts_token_count': 101, 'total_token_count': 215, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -1.5416392220391169, 'invocation_id': 'e-a6c375cd-84ff-4271-a68c-6ae6a3f0d6c4', 'author': 'greeter_agent', 'actions': {'state_delta': {'greeter_output': "Understood! I'll research good things to do for a weekend trip to Denver."}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'node_info': {'path': ''}, 'id': '6ad74a88-1f9a-434d-9252-fac4c1206958', 'timestamp': 1787613778.34791}

EVENT 2:
{'m